# Queen Editor Export → Hareket Prompt'u (Grok)

Queen Editor'ün **Export** dosyasındaki foto prompt'larını video grafının istediği **hareket prompt'una** çevirir. Türkçe → İngilizce çevirisi de aynı çağrıda olur. GPU gerekmez.

```
Queen Editor ──Export──> dugun-export.json   (bilgisayarına iner)
                              │  bu notebook'a yükle
                     [ bu notebook ]  ── her çeviriden sonra Drive'a yazar
                              │  indir butonu
                        video.json   →   photo_to_video.ipynb
```

**Drive'da nereye yazar:** `MyDrive/queen-tools/<proje>/video.json` — proje adı export'un içindeki `folder` yolundan gelir, sen bir şey yazmazsın.

**Asıl kopya Drive'dakidir.** İndirilen dosya, video notebook'una vermek için alınmış bir kopyadır. Elle düzelttiğin bir prompt'un kalıcı olmasını istiyorsan düzeltmeyi Drive'daki dosyada yap.

> **Yarıda kalırsa aynı export'u tekrar yükle.** `photo_prompt` alanı olan kareler çevrilmiştir, atlanır; kalanlar üretilir. Colab ölse de kayıp yok, ilerleme Drive'da.

> **Tek kareyi yeniden çevirtmek** için Drive'daki dosyada o satırın `photo_prompt` alanını sil.

**Gerekenler:** Colab **Secrets**'ta `XAI_API_KEY` (notebook erişimi açık).

## 1) CONFIG

Google Drive **burada** mount edilir: auth istemi ilk saniyede çıksın.

`INSTRUCTION` Grok'a verilen çeviri talimatı. Dolu geliyor: Wan I2V için kamerayı sabit tutan, sahneyi yeniden tarif etmeyip **hareketi** yazan bir talimat. Sonuçları gördükçe düzelteceğin asıl yer burası — talimat değişince yeniden çevirtmek için Drive'daki `video.json`'u sil.

**Boş bırakılırsa hücre durur**: boş talimatla istek atmak para harcayıp anlamsız cevap almaktır.

`XAI_API_KEY` Colab Secrets'tan okunur ve hiçbir çıktıya basılmaz.

In [ ]:
# === Google Drive — en başta mount edilir ===
# The auth prompt has to appear in the first second, not in the middle of a run.
from google.colab import drive, userdata
drive.mount('/content/drive')

# === Çeviri talimatı ===
# Sent as the system message; the photo prompt itself is the user message. It asks for one motion
# prompt as plain text because this notebook sends one request per frame -- a request that carried
# the whole list would be capped by the model's output limit, and a list-shaped answer would add a
# format to parse for no gain.
INSTRUCTION = """
You are an expert prompt engineer specializing in image-to-video generation with the Wan model.
I will give you one SDXL prompt that was used to generate a still image. Convert it into an
optimized Wan image-to-video (I2V) positive prompt.

Follow these rules:

Don't re-describe the static scene in detail — Wan already receives the actual image as input. The
image defines the appearance; your job is to define motion.
Keep the camera static — no camera movement, no zoom, no pan.
Focus primarily on the action in the image — bring the subject's main activity to life as natural,
continuous movement. Build the motion around what the subject is actively doing.
Add subtle secondary motion to support the main action (hair, clothing, breathing, environmental
details like wind or water).
Keep it natural and physically plausible — realistic motion looks better than exaggerated movement
that breaks the image.
Specify pacing and mood.

Output only the motion prompt itself, as one concise paragraph of plain text. No list, no
surrounding quotes, no numbering, no explanations, no markdown code fences, no extra text.
"""

# === xAI ===
# grok-3 retired on 2026-08-15 and its requests were already being routed here, so this is the
# same behaviour under its real name.
XAI_MODEL   = "grok-4.3"
XAI_URL     = "https://api.x.ai/v1/chat/completions"
XAI_TIMEOUT = 120                       # seconds per request

# === Drive ===
QUEEN_TOOLS_ROOT = "/content/drive/MyDrive/queen-tools"

import os
os.makedirs(QUEEN_TOOLS_ROOT, exist_ok=True)

# Colab Secrets: the key never appears in the notebook source or in any output.
XAI_API_KEY = userdata.get('XAI_API_KEY')

assert INSTRUCTION.strip(), "❌ INSTRUCTION boş — çeviri talimatını yaz (boş talimat = boşa harcanan istek)"
assert XAI_API_KEY, "❌ XAI_API_KEY okunamadı — Colab Secrets'a ekle ve 'Notebook access' aç"

print(f"✓ Model: {XAI_MODEL}")
print(f"✓ Drive kökü: {QUEEN_TOOLS_ROOT}")
print(f"✓ Talimat: {len(INSTRUCTION.strip())} karakter")
print(f"✓ Anahtar: Secrets'tan okundu ({len(XAI_API_KEY)} karakter)")

## 2) Export dosyasını yükle

Queen Editor'de **Export**'a basınca inen dosyayı buraya yükle. **İş emri yüklediğin dosyadır**: hangi projenin hangi kareleri isteniyor onu bu dosya söyler, Drive'daki dosya yalnız neyin bitmiş olduğunu söyler.

Dosya elle de düzenlenebildiği için biçimi burada kontrol edilir; eksik alan varsa hücre neyin eksik olduğunu yazıp durur.

In [ ]:
# === Export dosyasını yükle + biçimini doğrula ===
from google.colab import files
import json, os

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError(f"❌ Tek dosya yükle — {len(uploaded)} dosya geldi: {list(uploaded)}")

EXPORT_NAME = next(iter(uploaded))
try:
    EXPORT = json.loads(uploaded[EXPORT_NAME].decode("utf-8"))
except (UnicodeDecodeError, json.JSONDecodeError) as e:
    raise RuntimeError(f"❌ {EXPORT_NAME} okunamadı: {type(e).__name__}: {e}") from None

def validate_export(data):
    """Fail-loud on anything that is not a Queen Editor export. The file can be hand-edited and
    can come back from a chat window, so its shape is checked rather than trusted."""
    if not isinstance(data, dict):
        raise RuntimeError(f"❌ JSON bir nesne değil: {type(data).__name__}")
    for key in ("folder", "photos"):
        if key not in data:
            raise RuntimeError(f"❌ '{key}' alanı yok — dosyadaki anahtarlar: {sorted(data)}")
    if not isinstance(data["photos"], list):
        raise RuntimeError(f"❌ 'photos' liste değil: {type(data['photos']).__name__}")
    for i, photo in enumerate(data["photos"]):
        if not isinstance(photo, dict):
            raise RuntimeError(f"❌ photos[{i}] nesne değil: {type(photo).__name__}")
        missing = [k for k in ("file", "prompt") if k not in photo]
        if missing:
            raise RuntimeError(f"❌ photos[{i}] eksik alan: {missing} — var olanlar: {sorted(photo)}")

validate_export(EXPORT)

# The project name comes from the export's own folder path, so it is never typed by hand and can
# never point at a different project than the photos do.
PROJECT     = os.path.basename(EXPORT["folder"].rstrip("/"))
PROJECT_DIR = f"{QUEEN_TOOLS_ROOT}/{PROJECT}"
VIDEO_JSON  = f"{PROJECT_DIR}/video.json"
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"✓ {EXPORT_NAME}: {len(EXPORT['photos'])} kare")
print(f"✓ Proje: {PROJECT}")
print(f"✓ Fotoğrafların klasörü: {EXPORT['folder']}")
print(f"✓ Yazılacak dosya: {VIDEO_JSON}")

## 3) Birleştir + plan

Drive'da önceki koşudan bir dosya varsa açılır, ama **liste yüklediğin export'tan gelir**: export'a yeni eklenen kareler listeye girer, export'tan silinenler düşer, daha önce çevrilmiş olanlar (elle düzelttiklerin dahil) aynen korunur.

Aşağıdaki tablo hangi karenin çevrileceğini, hangisinin neden atlanacağını **tek bir istek atılmadan** gösterir.

In [ ]:
# === Birleştir + plan ===
# The plan is printed before a single request is paid for: a stale file or an already-complete
# project shows up here, not on the invoice.
import json, os

def load_done(path):
    """{file: row} from a previous run. A row counts as done only when it carries photo_prompt --
    that field is written in the same step as the translation, so it cannot be true early."""
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        previous = json.load(f)
    return {p["file"]: p for p in previous.get("photos", []) if p.get("photo_prompt")}

def merge(export, done):
    """The export's photo list, in its own order, carrying over translations we already have.
    A photo the export no longer lists is dropped: the export is what exists."""
    rows = []
    for photo in export["photos"]:
        carried = done.get(photo["file"])
        rows.append(dict(carried) if carried else {"file": photo["file"], "prompt": photo["prompt"]})
    return rows

def action_of(row):
    """(action, reason) for one row -- one place decides, so the table and the loop agree."""
    if row.get("photo_prompt"):
        return "ATLA", "zaten çevrildi"
    if not row["prompt"].strip():
        return "ATLA", "prompt boş"
    return "ÇEVİR", ""

DONE = load_done(VIDEO_JSON)
PLAN = merge(EXPORT, DONE)

print(f"\n{'DOSYA':<14}  {'KARAR':<6}  AÇIKLAMA")
print("-" * 74)
for row in PLAN:
    action, reason = action_of(row)
    detail = reason if reason else row["prompt"].strip().replace("\n", " ")[:44]
    print(f"{row['file']:<14}  {action:<6}  {detail}")

# Nothing left to translate is a valid state, not an error: the run simply spends nothing and the
# file is downloaded as it is.
_todo = sum(1 for row in PLAN if action_of(row)[0] == "ÇEVİR")
print("-" * 74)
print(f"Çevrilecek: {_todo}  |  Atlanacak: {len(PLAN) - _todo}")
if DONE:
    print(f"{len(DONE)} kare önceki koşudan geliyor: {VIDEO_JSON}")

## 4) Çevir

Kare başına bir istek gider, cevap düz metin olarak alınır. **Her başarılı çeviriden sonra dosya Drive'a yazılır** — runtime ölse bile bir sonraki koşu kaldığı yerden devam eder.

Bir istek patlarsa hücre durur ve o ana kadar çevrilenler Drive'da kalır; notebook'u tekrar çalıştırmak kaldığı yerden devam ettirir. Yeniden deneme yoktur, çünkü tekrar çalıştırmak zaten odur.

Sonunda eski/yeni prompt tablosu basılır — gözle geçir.

In [ ]:
# === Çevir ===
import json, os, time, requests

def save(path, folder, rows):
    """Rewrite the whole file. It is small and has one writer, and what has to survive is the
    run's progress -- not the number of writes."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"folder": folder, "photos": rows}, f, ensure_ascii=False, indent=2)

def to_motion_prompt(photo_prompt):
    """One photo prompt -> one motion prompt. The answer is plain text: at one prompt per call,
    asking for JSON would add a shape to negotiate for no gain."""
    response = requests.post(
        XAI_URL,
        headers={"Authorization": f"Bearer {XAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": XAI_MODEL,
              "messages": [{"role": "system", "content": INSTRUCTION},
                           {"role": "user",   "content": photo_prompt}]},
        timeout=XAI_TIMEOUT,
    )
    if response.status_code >= 400:
        # The service's own answer, verbatim -- a cause is never invented.
        raise RuntimeError(f"❌ xAI HTTP {response.status_code}\n{response.text}")
    try:
        text = response.json()["choices"][0]["message"]["content"].strip()
    except (json.JSONDecodeError, KeyError, IndexError, TypeError) as e:
        raise RuntimeError(f"❌ xAI cevabı beklenen biçimde değil ({type(e).__name__})\n"
                           f"{response.text}") from None
    if not text:
        raise RuntimeError(f"❌ xAI boş cevap döndü:\n{response.text}")
    return text

translated = 0
t_start = time.time()

for row in PLAN:
    if action_of(row)[0] != "ÇEVİR":
        continue
    print(f"  {row['file']}: çevriliyor…")
    try:
        motion = to_motion_prompt(row["prompt"])
    except RuntimeError as e:
        print(e)
        raise RuntimeError(
            f"❌ {row['file']} çevrilemedi — {translated} kare Drive'a yazıldı. "
            f"Aynı export'u tekrar yükle, kaldığı yerden devam eder."
        ) from None
    # photo_prompt is written in the same step as the translation: that is what makes it a
    # trustworthy "already done" marker for the next run.
    row["photo_prompt"] = row["prompt"]
    row["prompt"] = motion
    save(VIDEO_JSON, EXPORT["folder"], PLAN)
    translated += 1

save(VIDEO_JSON, EXPORT["folder"], PLAN)
print(f"\n✅ {translated} kare çevrildi ({time.time() - t_start:.0f} sn) → {VIDEO_JSON}")

print(f"\n{'DOSYA':<14}  {'ESKİ (foto)':<36}  YENİ (hareket)")
print("-" * 100)
for row in PLAN:
    old = (row.get("photo_prompt") or row["prompt"]).strip().replace("\n", " ")[:34]
    new = row["prompt"].strip().replace("\n", " ")[:44] if row.get("photo_prompt") else "—"
    print(f"{row['file']:<14}  {old:<36}  {new}")

## 5) İndir

Dosya Drive'da zaten duruyor; bu hücre onu bilgisayarına indirir ki `photo_to_video.ipynb`'a yükleyebilesin.

In [ ]:
# === İndir ===
# The Drive copy stays the master (hand edits belong there); this is the copy that gets handed to
# the video notebook.
from google.colab import files
files.download(VIDEO_JSON)